In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/vit-huge-plus-patch16-dinov3-lvd1689m/pytorch/default/1/vit_huge_plus_patch16_dinov3.lvd1689m_backbone.pth
/kaggle/input/pytorch-efficientnet-imagenet1k_v1-rsna-2024/pytorch/fulldata-20epoch-aug-extalayers/1/model_20240909_205101.pth
/kaggle/input/speciesnet/pytorch/v4.0.2b/1/full_image_88545560_22x8_v12_epoch_00153.labels.20251208.txt
/kaggle/input/speciesnet/pytorch/v4.0.2b/1/README.md
/kaggle/input/speciesnet/pytorch/v4.0.2b/1/info.json
/kaggle/input/speciesnet/pytorch/v4.0.2b/1/taxonomy_release.20251208.txt
/kaggle/input/speciesnet/pytorch/v4.0.2b/1/full_image_88545560_22x8_v12_epoch_00153.pt
/kaggle/input/speciesnet/pytorch/v4.0.2b/1/geofence_release.20251208.json
/kaggle/input/mask-rcnn-models/pytorch/default/10/MaskrcnnMobileNetV2_best.pt
/kaggle/input/mask-rcnn-models/pytorch/default/10/Maskrcnn_best.pt
/kaggle/input/se_resnet50/pytorch/default/1/se_resnet50-ce0d4300.pth
/kaggle/input/efficient-net-v2/pytorch/default/1/efficientnet_v2_s-dd5fe13b.pth
/kaggle/input/

In [2]:
# ============================================================================
# CELL 1: Setup and Data Loading
# ============================================================================
import numpy as np
import pandas as pd
import os
import torch
import torchvision
import torch.nn as nn
import pytorch_lightning as pl
import torchvision.models as models
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from torchvision.transforms import v2
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from pytorch_lightning import Trainer
from tqdm import tqdm
import cv2
from pytorch_lightning.callbacks import EarlyStopping, ModelCheckpoint

# Load data
train_file_path = "/kaggle/input/csiro-biomass/train.csv"
test_file_path = "/kaggle/input/csiro-biomass/test.csv"

train_pd = pd.read_csv(train_file_path)
test_pd_original = pd.read_csv(test_file_path)  # Keep original for submission
test_pd = test_pd_original.copy()  # Working copy for predictions

print(f"Train shape: {train_pd.shape}")
print(f"Test shape: {test_pd.shape}")

Train shape: (1785, 9)
Test shape: (5, 3)


In [3]:
# ============================================================================
# CELL 2: HEIGHT PREDICTION
# ============================================================================
print("\n=== STEP 1: Height Prediction ===")

height_model = torchvision.models.detection.maskrcnn_resnet50_fpn(weights=None, weights_backbone=None)
path = "/kaggle/input/csiro-biomass"
local_weights = "/kaggle/input/mask-rcnn-models/pytorch/default/10/Maskrcnn_best.pt"

try:
    state_dict = torch.load(local_weights, map_location="cpu", weights_only=False)
    height_model = state_dict
    height_model.eval()
    
    for index, image_path in test_pd["image_path"].items():
        image_path = os.path.join(path, image_path)
        image = Image.open(image_path).convert("RGB")
        
        image_transform = v2.Compose([
            v2.Resize((224, 224)),
            v2.ToImage(),
            v2.ToDtype(torch.float32, scale=True),
            v2.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
        ])
        
        image_tensor = image_transform(image)
        
        with torch.no_grad():
            outputs = height_model([image_tensor])
        
        output_pixels = outputs[0]["boxes"].cpu().numpy()
        x1, y1, x2, y2 = output_pixels[0]
        image_in_cm = y2 / 2.54
        test_pd.loc[index, "Height_Ave_cm"] = image_in_cm
    
    print("✅ Height predictions completed using Mask R-CNN")
except:
    # Fallback to mean height
    mean_height = train_pd["Height_Ave_cm"].mean()
    test_pd["Height_Ave_cm"] = mean_height
    print(f"✅ Height predictions using mean: {mean_height:.2f}")


=== STEP 1: Height Prediction ===
✅ Height predictions completed using Mask R-CNN


species model

In [4]:
import torch

torch.cuda.empty_cache()

In [5]:
# ============================================================================
# CELL 3: SPECIES PREDICTION (EFFICIENTNET-B6)
# ============================================================================
import os
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models
from torchvision.transforms import v2
import pytorch_lightning as pl
from pytorch_lightning import Trainer
from pytorch_lightning.callbacks import EarlyStopping, ModelCheckpoint
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from PIL import Image
from tqdm import tqdm

print("\n=== STEP 2: Species Prediction (EfficientNet-B6) ===")

# --- Global Encoders ---
SPECIES_LE = LabelEncoder()
TARGET_LE = LabelEncoder()

# Fit encoders
SPECIES_LE.fit(train_pd["Species"].astype(str).unique())
TARGET_LE.fit(train_pd["target_name"].astype(str).unique())

def safe_encode(le, val):
    val_str = str(val)
    if val_str in le.classes_:
        return le.transform([val_str])[0]
    return 0

# --- Dataset Class ---
class SpeciesDataset(Dataset):
    def __init__(self, df, root_dir, transform=None):
        self.df = df.reset_index(drop=True)
        self.root_dir = root_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image_path = os.path.join(self.root_dir, row["image_path"])
        
        try:
            image = Image.open(image_path).convert("RGB")
        except:
            image = Image.new('RGB', (224, 224), (0, 0, 0))
            
        tabular = torch.tensor([
            float(row["target_name"]), 
            float(row["Height_Ave_cm"])
        ], dtype=torch.float32)
        
        y = torch.tensor(int(row["Species"]), dtype=torch.long)
        
        if self.transform:
            image = self.transform(image)
        return image, tabular, y

# --- Data Module ---
class SpeciesDataModule(pl.LightningDataModule):
    def __init__(self, train_df, valid_df, root_dir, batch_size=16, num_workers=2):
        super().__init__()
        self.train_df, self.valid_df = train_df.copy(), valid_df.copy()
        self.root_dir, self.batch_size, self.num_workers = root_dir, batch_size, num_workers

    def setup(self, stage=None):
        for df in [self.train_df, self.valid_df]:
            if not np.issubdtype(df["Species"].dtype, np.number):
                df["Species"] = df["Species"].apply(lambda x: safe_encode(SPECIES_LE, x))
            if not np.issubdtype(df["target_name"].dtype, np.number):
                df["target_name"] = df["target_name"].apply(lambda x: safe_encode(TARGET_LE, x))

        self.train_tf = v2.Compose([
            v2.RandomResizedCrop(380), # B6 likes larger images, usually 528, but 380 is good for speed/memory
            v2.RandomHorizontalFlip(), 
            v2.ToImage(),
            v2.ToDtype(torch.float32, scale=True),
            v2.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
        ])
        self.valid_tf = v2.Compose([
            v2.Resize(400), 
            v2.CenterCrop(380), 
            v2.ToImage(),
            v2.ToDtype(torch.float32, scale=True),
            v2.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
        ])
        
        self.train_ds = SpeciesDataset(self.train_df, self.root_dir, self.train_tf)
        self.valid_ds = SpeciesDataset(self.valid_df, self.root_dir, self.valid_tf)

    def train_dataloader(self):
        return DataLoader(self.train_ds, batch_size=self.batch_size, shuffle=True, num_workers=self.num_workers)
    
    def val_dataloader(self):
        return DataLoader(self.valid_ds, batch_size=self.batch_size, shuffle=False, num_workers=self.num_workers)

# --- Classifier Model (EfficientNet-B6) ---
class SpeciesClassifier(pl.LightningModule):
    def __init__(self, num_classes, target_dim, lr=1e-4):
        super().__init__()
        self.save_hyperparameters()
        
        # 1. Initialize Architecture
        print("Initializing EfficientNet-B6...")
        self.base_model = models.efficientnet_b6(weights=None)
        
        # 2. Load Weights
        weights_path = "/kaggle/input/efficientnet-b6/pytorch/default/1/efficientnet_b6_lukemelas-24a108a5.pth"
        print(f"Loading weights from: {weights_path}")
        
        try:
            state_dict = torch.load(weights_path, weights_only=True)
            # 'strict=False' is important here because PyTorch versions can vary slightly 
            # in naming convention (e.g. 'features' vs 'blocks')
            self.base_model.load_state_dict(state_dict, strict=False)
            print("✅ Weights loaded successfully (strict=False).")
        except Exception as e:
            print(f"⚠️ Warning: Could not load weights: {e}")
            print("Continuing with random initialization (Training will be harder).")

        # 3. Get Feature Dimension & Remove Head
        # EfficientNet-B6 classifier block is usually: Sequential(Dropout, Linear)
        # We need the in_features of that Linear layer.
        self.img_dim = self.base_model.classifier[1].in_features 
        self.base_model.classifier = nn.Identity()
        
        print(f"✅ Model Ready. Feature Dim: {self.img_dim}")

        # 4. Tabular & Fusion Heads
        self.target_emb = nn.Embedding(target_dim + 1, 8)
        self.tabular_net = nn.Sequential(
            nn.Linear(8 + 1, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.3)
        )
        
        self.head = nn.Sequential(
            nn.Linear(self.img_dim + 64, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(256, num_classes)
        )
        
        self.loss_fn = nn.CrossEntropyLoss(label_smoothing=0.1)

    def forward(self, img, target_name, height):
        img_feats = self.base_model(img)
        # EfficientNet outputs are already flattened by the pooling layer before the classifier
        if len(img_feats.shape) > 2:
            img_feats = img_feats.view(img_feats.size(0), -1)
            
        t_feat = self.target_emb(target_name.long())
        tab_in = torch.cat([t_feat, height.unsqueeze(1)], dim=1)
        tab_feats = self.tabular_net(tab_in)
        
        combined = torch.cat([img_feats, tab_feats], dim=1)
        return self.head(combined)

    def training_step(self, batch, batch_idx):
        img, tab, y = batch
        logits = self(img, tab[:, 0], tab[:, 1])
        loss = self.loss_fn(logits, y)
        acc = (logits.argmax(dim=1) == y).float().mean()
        self.log("train_loss", loss, prog_bar=True)
        self.log("train_acc", acc, prog_bar=True)
        return loss

    def validation_step(self, batch, batch_idx):
        img, tab, y = batch
        logits = self(img, tab[:, 0], tab[:, 1])
        loss = self.loss_fn(logits, y)
        acc = (logits.argmax(dim=1) == y).float().mean()
        self.log("val_loss", loss, prog_bar=True)
        self.log("val_acc", acc, prog_bar=True)

    def configure_optimizers(self):
        optimizer = torch.optim.AdamW(self.parameters(), lr=self.hparams.lr, weight_decay=0.01)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
            optimizer, T_0=5, T_mult=1, eta_min=1e-6
        )
        return [optimizer], [scheduler]

# --- Training Section ---
image_root_dir = "/kaggle/input/csiro-biomass"

train_df, valid_df = train_test_split(
    train_pd, 
    test_size=0.2, 
    random_state=42, 
    stratify=train_pd["Species"]
)

# Reduced batch size slightly as B6 is VRAM heavy
datamodule = SpeciesDataModule(train_df=train_df, valid_df=valid_df, root_dir=image_root_dir, batch_size=8)
datamodule.setup()

num_species = len(SPECIES_LE.classes_)
target_count = len(TARGET_LE.classes_)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

species_model = SpeciesClassifier(
    num_classes=num_species, 
    target_dim=target_count
).to(device)

early_stop_callback = EarlyStopping(monitor="val_loss", patience=5, mode="min")
checkpoint_callback = ModelCheckpoint(monitor="val_loss", save_top_k=1, mode="min")

trainer = Trainer(
    accelerator="gpu",
    devices=1,
    max_epochs=1, # 20 epochs is usually sufficient for fine-tuning
    precision="16-mixed",
    gradient_clip_val=1.0, 
    callbacks=[early_stop_callback, checkpoint_callback]
)

trainer.fit(species_model, datamodule)

# --- Inference Section ---
print("\n=== Starting Species Inference ===")
species_model.eval()
species_model.to(device)

inf_tf = v2.Compose([
    v2.Resize(400), v2.CenterCrop(380), v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

species_results = []

with torch.no_grad():
    for _, row in tqdm(test_pd.iterrows(), total=len(test_pd), desc="Predicting Species"):
        try:
            img_path = os.path.join(image_root_dir, row["image_path"])
            img = Image.open(img_path).convert("RGB")
            img_tensor = inf_tf(img).unsqueeze(0).to(device)
            
            t_idx = safe_encode(TARGET_LE, row["target_name"])
            tab_tensor = torch.tensor(
                [[t_idx, float(row["Height_Ave_cm"])]], 
                dtype=torch.float32
            ).to(device)
            
            logits = species_model(img_tensor, tab_tensor[:, 0], tab_tensor[:, 1])
            class_idx = logits.argmax(dim=1).item()
            actual_name = SPECIES_LE.inverse_transform([class_idx])[0]
            species_results.append(actual_name)
        except Exception as e:
            species_results.append(SPECIES_LE.classes_[0])

test_pd["Species"] = species_results
print("✅ Species predictions completed")


=== STEP 2: Species Prediction (EfficientNet-B6) ===
Initializing EfficientNet-B6...
Loading weights from: /kaggle/input/efficientnet-b6/pytorch/default/1/efficientnet_b6_lukemelas-24a108a5.pth
✅ Weights loaded successfully (strict=False).
✅ Model Ready. Feature Dim: 2304


Using 16bit Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
2026-01-18 11:23:37.693166: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1768735417.876287      24 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1768735417.928817      24 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1768735418.370302      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1768735418.370330      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid lin

┏━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name        ┃ Type             ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ base_model  │ EfficientNet     │ 40.7 M │ train │     0 │
│ 1 │ target_emb  │ Embedding        │     48 │ train │     0 │
│ 2 │ tabular_net │ Sequential       │    768 │ train │     0 │
│ 3 │ head        │ Sequential       │  610 K │ train │     0 │
│ 4 │ loss_fn     │ CrossEntropyLoss │      0 │ train │     0 │
└───┴─────────────┴──────────────────┴────────┴───────┴───────┘

Trainable params: 41.3 M                                                                                           
Non-trainable params: 0                                                                                            
Total params: 41.3 M                                                                                               
Total estimated model params size (MB): 165                                                                        
Modules in train mode: 920                                                                                         
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

`Trainer.fit` stopped: `max_epochs=1` reached.



=== Starting Species Inference ===


Predicting Species: 100%|██████████| 5/5 [00:00<00:00, 10.09it/s]

✅ Species predictions completed


In [6]:
import torch

torch.cuda.empty_cache()

In [7]:
# ============================================================================
# CELL 4: STATE PREDICTION (EfficientNet-B6 Version)
# ============================================================================
import os
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models
from torchvision.transforms import v2
import pytorch_lightning as pl
from pytorch_lightning import Trainer
from pytorch_lightning.callbacks import EarlyStopping, ModelCheckpoint
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from PIL import Image
from tqdm import tqdm

print("\n=== STEP 3: State Prediction (Custom Model: EfficientNet-B6) ===")

# --- Global Encoder for State ---
STATE_LE = LabelEncoder()
# Fit on train data
STATE_LE.fit(train_pd["State"].astype(str).unique())

# (Reusing SPECIES_LE and TARGET_LE from previous steps, assuming they exist globally)

# --- Dataset Class ---
class StateDataset(Dataset):
    def __init__(self, df, root_dir, transform=None):
        self.df = df.reset_index(drop=True)
        self.root_dir = root_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image_path = os.path.join(self.root_dir, row["image_path"])
        
        try:
            image = Image.open(image_path).convert("RGB")
        except:
            image = Image.new('RGB', (224, 224), (0, 0, 0))

        # Tabular: [target_name, height, species]
        # We use the predicted 'Species' column in test, or actual in train
        tabular = torch.tensor([
            float(row["target_name"]), 
            float(row["Height_Ave_cm"]), 
            float(row["Species"])
        ], dtype=torch.float32)

        # Target Label: State
        y = torch.tensor(int(row["State"]), dtype=torch.long)
        
        if self.transform:
            image = self.transform(image)
        return image, tabular, y

# --- Data Module ---
class StateDataModule(pl.LightningDataModule):
    def __init__(self, train_df, valid_df, root_dir, batch_size=16, num_workers=2):
        super().__init__()
        self.train_df, self.valid_df = train_df.copy(), valid_df.copy()
        self.root_dir, self.batch_size, self.num_workers = root_dir, batch_size, num_workers

    def setup(self, stage=None):
        for df in [self.train_df, self.valid_df]:
            # Safe Encode State
            if not np.issubdtype(df["State"].dtype, np.number):
                df["State"] = df["State"].apply(lambda x: safe_encode(STATE_LE, x))
            # Safe Encode Target Name
            if not np.issubdtype(df["target_name"].dtype, np.number):
                df["target_name"] = df["target_name"].apply(lambda x: safe_encode(TARGET_LE, x))
            # Safe Encode Species (This is an input feature for State prediction)
            if not np.issubdtype(df["Species"].dtype, np.number):
                df["Species"] = df["Species"].apply(lambda x: safe_encode(SPECIES_LE, x))

        self.train_tf = v2.Compose([
            v2.RandomResizedCrop(224), v2.RandomHorizontalFlip(), v2.ToImage(),
            v2.ToDtype(torch.float32, scale=True),
            v2.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
        ])
        self.valid_tf = v2.Compose([
            v2.Resize(256), v2.CenterCrop(224), v2.ToImage(),
            v2.ToDtype(torch.float32, scale=True),
            v2.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
        ])
        self.train_ds = StateDataset(self.train_df, self.root_dir, self.train_tf)
        self.valid_ds = StateDataset(self.valid_df, self.root_dir, self.valid_tf)

    def train_dataloader(self):
        return DataLoader(self.train_ds, batch_size=self.batch_size, shuffle=True, num_workers=self.num_workers)
    
    def val_dataloader(self):
        return DataLoader(self.valid_ds, batch_size=self.batch_size, shuffle=False, num_workers=self.num_workers)

# --- Classifier Model (Updated for EfficientNet-B6) ---
class StateClassifier(pl.LightningModule):
    def __init__(self, num_classes, target_dim, species_dim, lr=1e-4):
        super().__init__()
        self.save_hyperparameters()
        
        # --- LOADING EFFICIENTNET-B6 ---
        weights_path = "/kaggle/input/efficientnet-b6/pytorch/default/1/efficientnet_b6_lukemelas-24a108a5.pth"
        print(f"Loading backbone from: {weights_path}")
        
        try:
            # 1. Initialize empty architecture
            self.base_model = models.efficientnet_b6(weights=None)
            
            # 2. Load weights
            state_dict = torch.load(weights_path, map_location='cpu')
            
            # 3. Handle key mismatch (if strict=False needed)
            self.base_model.load_state_dict(state_dict)
            print("✅ Successfully loaded EfficientNet-B6 weights.")
            
        except Exception as e:
            print(f"⚠️ Error loading custom weights: {e}. Downloading standard weights instead.")
            self.base_model = models.efficientnet_b6(weights='DEFAULT')

        # --- DYNAMIC HEAD STRIPPING ---
        # EfficientNet uses 'classifier' usually.
        if hasattr(self.base_model, 'classifier'):
             # Get input features of the last layer (often a Sequential with Dropout+Linear)
             if isinstance(self.base_model.classifier, nn.Sequential):
                 self.img_dim = self.base_model.classifier[-1].in_features
             else:
                 self.img_dim = self.base_model.classifier.in_features
             
             # Remove the head
             self.base_model.classifier = nn.Identity()
             
        elif hasattr(self.base_model, 'fc'):
            self.img_dim = self.base_model.fc.in_features
            self.base_model.fc = nn.Identity()
            
        else:
            print("⚠️ Could not detect head. Assuming dim=2304 (Std EffNet-B6).")
            self.img_dim = 2304

        print(f"Feature Dimension: {self.img_dim}")

        # Embeddings for Categorical Inputs
        self.target_emb = nn.Embedding(target_dim + 1, 8)
        self.species_emb = nn.Embedding(species_dim + 1, 12) # Species helps predict State
        
        # Tabular Fusion Layer
        # Inputs: TargetEmb(8) + Height(1) + SpeciesEmb(12) = 21
        self.tabular_net = nn.Sequential(
            nn.Linear(21, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.3)
        )
        
        # Final Classification Head
        self.head = nn.Sequential(
            nn.Linear(self.img_dim + 64, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(512, num_classes)
        )
        
        self.loss_fn = nn.CrossEntropyLoss(label_smoothing=0.1)

    def forward(self, img, target_name, height, species):
        # Image Features
        img_feats = self.base_model(img)
        if len(img_feats.shape) > 2:
            img_feats = img_feats.view(img_feats.size(0), -1)
        
        # Tabular Features
        t_feat = self.target_emb(target_name.long())
        s_feat = self.species_emb(species.long())
        
        # Concat: [Target, Height, Species]
        tab_in = torch.cat([t_feat, height.unsqueeze(1), s_feat], dim=1)
        tab_feats = self.tabular_net(tab_in)
        
        # Final Concat
        combined = torch.cat([img_feats, tab_feats], dim=1)
        return self.head(combined)

    def training_step(self, batch, batch_idx):
        img, tab, y = batch
        # tab order: [target_name, height, species]
        logits = self(img, tab[:, 0], tab[:, 1], tab[:, 2])
        loss = self.loss_fn(logits, y)
        acc = (logits.argmax(dim=1) == y).float().mean()
        self.log("train_loss", loss, prog_bar=True)
        self.log("train_acc", acc, prog_bar=True)
        return loss

    def validation_step(self, batch, batch_idx):
        img, tab, y = batch
        logits = self(img, tab[:, 0], tab[:, 1], tab[:, 2])
        loss = self.loss_fn(logits, y)
        acc = (logits.argmax(dim=1) == y).float().mean()
        self.log("val_loss", loss, prog_bar=True)
        self.log("val_acc", acc, prog_bar=True)

    def configure_optimizers(self):
        optimizer = torch.optim.AdamW(self.parameters(), lr=self.hparams.lr, weight_decay=0.05)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10)
        return [optimizer], [scheduler]


# --- Training Section ---
# Ensure stratification by State
train_df, valid_df = train_test_split(train_pd, test_size=0.2, random_state=42, stratify=train_pd["State"])

datamodule = StateDataModule(train_df=train_df, valid_df=valid_df, root_dir=image_root_dir)
datamodule.setup()

# Dimensions
num_states = len(STATE_LE.classes_)
target_count = len(TARGET_LE.classes_)
species_count = len(SPECIES_LE.classes_)

# Initialize Model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
state_model = StateClassifier(
    num_classes=num_states, 
    target_dim=target_count, 
    species_dim=species_count
).to(device)

# Callbacks
early_stop_callback = EarlyStopping(monitor="val_loss", patience=5, mode="min")
checkpoint_callback = ModelCheckpoint(monitor="val_loss", save_top_k=1, mode="min")

# Trainer
trainer = Trainer(
    accelerator="gpu",
    devices=1,
    max_epochs=1,
    precision="16-mixed",
    gradient_clip_val=1.0, 
    callbacks=[early_stop_callback, checkpoint_callback]
)

trainer.fit(state_model, datamodule)

# --- Inference Section ---
state_model.eval()
state_model.to(device)

state_results = []

# Using the inference transform
inf_tf = v2.Compose([
    v2.Resize(256), v2.CenterCrop(224), v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

with torch.no_grad():
    for _, row in tqdm(test_pd.iterrows(), total=len(test_pd), desc="Predicting State"):
        try:
            img_path = os.path.join(image_root_dir, row["image_path"])
            img = Image.open(img_path).convert("RGB")
            img_tensor = inf_tf(img).unsqueeze(0).to(device)
            
            # Prepare inputs
            t_idx = safe_encode(TARGET_LE, row["target_name"])
            s_idx = safe_encode(SPECIES_LE, row["Species"]) # Used predictions from prev step
            h_val = float(row["Height_Ave_cm"])
            
            # Tensors
            t_tensor = torch.tensor([t_idx], device=device)
            h_tensor = torch.tensor([h_val], device=device)
            s_tensor = torch.tensor([s_idx], device=device)
            
            logits = state_model(img_tensor, t_tensor, h_tensor, s_tensor)
            
            class_idx = logits.argmax(dim=1).item()
            actual_state = STATE_LE.inverse_transform([class_idx])[0]
            state_results.append(actual_state)
        except Exception as e:
            # Fallback
            state_results.append(STATE_LE.classes_[0])

test_pd["State"] = state_results
print("✅ State predictions completed")


=== STEP 3: State Prediction (Custom Model: EfficientNet-B6) ===
Loading backbone from: /kaggle/input/efficientnet-b6/pytorch/default/1/efficientnet_b6_lukemelas-24a108a5.pth


Using 16bit Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


✅ Successfully loaded EfficientNet-B6 weights.
Feature Dimension: 2304


/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/model_summary/model_summary.py:242: Precision 16-mixed is not supported by the model summary.  Estimated model size in MB will not be accurate. Using 32 bits instead.


┏━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name        ┃ Type             ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ base_model  │ EfficientNet     │ 40.7 M │ train │     0 │
│ 1 │ target_emb  │ Embedding        │     48 │ train │     0 │
│ 2 │ species_emb │ Embedding        │    192 │ train │     0 │
│ 3 │ tabular_net │ Sequential       │  1.5 K │ train │     0 │
│ 4 │ head        │ Sequential       │  1.2 M │ train │     0 │
│ 5 │ loss_fn     │ CrossEntropyLoss │      0 │ train │     0 │
└───┴─────────────┴──────────────────┴────────┴───────┴───────┘

Trainable params: 42.0 M                                                                                           
Non-trainable params: 0                                                                                            
Total params: 42.0 M                                                                                               
Total estimated model params size (MB): 167                                                                        
Modules in train mode: 921                                                                                         
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

`Trainer.fit` stopped: `max_epochs=1` reached.


Predicting State: 100%|██████████| 5/5 [00:00<00:00, 11.45it/s]

✅ State predictions completed


In [8]:
import torch

torch.cuda.empty_cache()

In [9]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import timm 
import os
import cv2
import numpy as np
import pytorch_lightning as pl
import gc
from torch.utils.data import Dataset, DataLoader
# from tqdm import tqdm  <-- Removed to prevent IOPub timeout
from torchvision.transforms import v2
from PIL import Image
from sklearn.model_selection import train_test_split
from pytorch_lightning.callbacks import EarlyStopping, ModelCheckpoint

# --- 0. Memory Management Setup ---
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
torch.cuda.empty_cache()
gc.collect()
print("\n=== STEP 4: NDVI Prediction (Memory Optimized & Log Fix) ===")

# --- 1. Global Setup & Configuration ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
image_root_dir = "/kaggle/input/csiro-biomass"
NDVI_IMG_SIZE = 768  
NDVI_BATCH_SIZE = 1  
ACCUMULATE_GRAD = 8  

def image_to_mask(image_path):
    image = cv2.imread(image_path)
    if image is None:
        return np.zeros((NDVI_IMG_SIZE, NDVI_IMG_SIZE, 3), dtype=np.uint8)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    hsv = cv2.cvtColor(image, cv2.COLOR_RGB2HSV)
    lower_green = np.array([35, 40, 40])
    upper_green = np.array([85, 255, 255])
    mask = cv2.inRange(hsv, lower_green, upper_green)
    return cv2.bitwise_and(image, image, mask=mask)

def safe_encode(encoder, value):
    try: return encoder.transform([value])[0]
    except: return 0

# --- 2. Transforms ---
ndvi_train_tfs = v2.Compose([
    v2.ToImage(),
    v2.Resize((NDVI_IMG_SIZE, NDVI_IMG_SIZE)),
    v2.RandomHorizontalFlip(p=0.5),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

ndvi_valid_tfs = v2.Compose([
    v2.ToImage(),
    v2.Resize((NDVI_IMG_SIZE, NDVI_IMG_SIZE)),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# --- 3. Dataset & Model ---
class PreGSSHDataset(Dataset):
    def __init__(self, df, root_dir, transform=None):
        self.df = df.reset_index(drop=True)
        self.root_dir = root_dir
        self.transform = transform

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = os.path.join(self.root_dir, row["image_path"])
        masked_img = image_to_mask(img_path)
        
        # Assuming SPECIES_LE and TARGET_LE are defined globally in previous cells
        s_idx = safe_encode(SPECIES_LE, row["Species"])
        t_idx = safe_encode(TARGET_LE, row["target_name"])
        
        ndvi_val = row["Pre_GSHH_NDVI"] if "Pre_GSHH_NDVI" in self.df.columns else 0.0
        
        tabular = torch.tensor([float(s_idx), float(row["Height_Ave_cm"]), float(t_idx)], dtype=torch.float32)
        target = torch.tensor([ndvi_val], dtype=torch.float32)
        
        if self.transform: masked_img = self.transform(masked_img)
        
        return masked_img, tabular, target

class PreGSSHModel(pl.LightningModule):
    def __init__(self, species_dim, target_dim, lr=1e-4):
        super().__init__()
        self.save_hyperparameters()
        
        # 1. FIXED Backbone
        try:
            self.backbone = timm.create_model(
                'vit_huge_patch14_224', 
                pretrained=False, 
                num_classes=0,
                img_size=NDVI_IMG_SIZE,
                patch_size=16 
            )
        except Exception as e:
            self.backbone = timm.create_model(
                'vit_huge_patch16_224', 
                pretrained=False, 
                num_classes=0,
                img_size=NDVI_IMG_SIZE
            )
        
        self.backbone.set_grad_checkpointing(True)
        
        # 2. Weight Loading
        backbone_path = "/kaggle/input/vit-huge-plus-patch16-dinov3-lvd1689m/pytorch/default/1/vit_huge_plus_patch16_dinov3.lvd1689m_backbone.pth"
        if os.path.exists(backbone_path):
            state_dict = torch.load(backbone_path, map_location='cpu')
            if 'state_dict' in state_dict: state_dict = state_dict['state_dict']
            elif 'model' in state_dict: state_dict = state_dict['model']
            
            clean_state_dict = {k.replace('backbone.', '').replace('model.', '').replace('_orig_mod.', ''): v 
                               for k, v in state_dict.items()}
            
            msg = self.backbone.load_state_dict(clean_state_dict, strict=False)
            print(f"✅ Loaded weights. Missing: {len(msg.missing_keys)}, Unexpected: {len(msg.unexpected_keys)}")
        
        # 3. Freeze Backbone
        for param in self.backbone.parameters():
            param.requires_grad = False
            
        self.img_dim = self.backbone.num_features 
        self.species_emb = nn.Embedding(species_dim + 1, 8)
        self.target_emb = nn.Embedding(target_dim + 1, 4)
        
        self.tabular_net = nn.Sequential(
            nn.Linear(13, 64),
            nn.LayerNorm(64),
            nn.SiLU(),
            nn.Dropout(0.2)
        )
        
        self.head = nn.Sequential(
            nn.Linear(self.img_dim + 64, 128),
            nn.SiLU(),
            nn.LayerNorm(128),
            nn.Linear(128, 1),
            nn.Sigmoid() 
        )
        self.loss_fn = nn.HuberLoss(delta=0.1)

    def forward(self, img, species, height, target_name):
        img_feats = self.backbone(img)
        if isinstance(img_feats, (list, tuple)): img_feats = img_feats[-1]
        if img_feats.dim() > 2: img_feats = img_feats.mean(dim=1)
            
        s_feat = self.species_emb(species.long())
        t_feat = self.target_emb(target_name.long())
        h_feat = height.unsqueeze(1) if height.dim() == 1 else height
        
        tab_in = torch.cat([s_feat, t_feat, h_feat], dim=1)
        tab_feats = self.tabular_net(tab_in)
        
        combined = torch.cat([img_feats, tab_feats], dim=1)
        return self.head(combined)

    def training_step(self, batch, batch_idx):
        img, tab, y = batch
        preds = self(img, tab[:, 0], tab[:, 1], tab[:, 2])
        loss = self.loss_fn(preds, y)
        # prog_bar=True forces updates to the progress bar, removed it to be safe
        self.log("train_loss", loss, prog_bar=False, on_step=False, on_epoch=True)
        return loss

    def validation_step(self, batch, batch_idx):
        img, tab, y = batch
        preds = self(img, tab[:, 0], tab[:, 1], tab[:, 2])
        mse = F.mse_loss(preds, y)
        self.log("val_mse", mse, prog_bar=False, on_step=False, on_epoch=True)

    def configure_optimizers(self):
        return torch.optim.AdamW(self.parameters(), lr=self.hparams.lr)

# --- 4. Execution ---
train_df, valid_df = train_test_split(train_pd, test_size=0.2, random_state=42)

train_loader = DataLoader(PreGSSHDataset(train_df, image_root_dir, ndvi_train_tfs), 
                          batch_size=NDVI_BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True, drop_last=True)
valid_loader = DataLoader(PreGSSHDataset(valid_df, image_root_dir, ndvi_valid_tfs), 
                          batch_size=NDVI_BATCH_SIZE, num_workers=2, drop_last=False)

ndvi_model = PreGSSHModel(species_dim=len(SPECIES_LE.classes_), target_dim=len(TARGET_LE.classes_))

# --- FIX APPLIED HERE ---
# enable_progress_bar=False prevents the IO flood
# log_every_n_steps=50 reduces update frequency
trainer = pl.Trainer(
    accelerator="gpu", 
    devices=1, 
    max_epochs=1, 
    precision="16-mixed",
    accumulate_grad_batches=ACCUMULATE_GRAD,
    callbacks=[
        EarlyStopping(monitor="val_mse", patience=3, verbose=False),
        ModelCheckpoint(monitor="val_mse", filename="best_ndvi", mode="min", verbose=False)
    ],
    log_every_n_steps=50,       # Increased from 5 to reduce logs
    enable_progress_bar=False,  # CRITICAL: Disables the crashing progress bar
    enable_model_summary=True   # Keep summary to see model structure
)

print("Starting Training (Progress bar disabled for stability)...")
trainer.fit(ndvi_model, train_loader, valid_loader)

# Clear memory
del train_loader, valid_loader
torch.cuda.empty_cache()
gc.collect()

# --- 5. Inference ---
print("Starting Inference...")
ndvi_model.eval().to(device)
ndvi_results = []
test_dataset = PreGSSHDataset(test_pd, image_root_dir, ndvi_valid_tfs)
test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False, num_workers=2)

with torch.no_grad():
    # Removed tqdm loop, using simple enumeration with occasional print
    for i, batch in enumerate(test_loader):
        img, tab, _ = batch
        img, tab = img.to(device), tab.to(device)
        pred = ndvi_model(img, tab[:, 0], tab[:, 1], tab[:, 2])
        ndvi_results.append(pred.squeeze().item())
        
        if i % 500 == 0:
            print(f"Processed {i} images...")

test_pd["Pre_GSHH_NDVI"] = ndvi_results
print("✅ Done")

/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` 


=== STEP 4: NDVI Prediction (Memory Optimized & Log Fix) ===


Using 16bit Automatic Mixed Precision (AMP)


✅ Loaded weights. Missing: 97, Unexpected: 193


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


...

/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/model_summary/model_summary.py:242: Precision 16-mixed is not supported by the model summary.  Estimated model size in MB will not be accurate. Using 32 bits instead.


┏━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name        ┃ Type              ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ backbone    │ VisionTransformer │  633 M │ train │     0 │
│ 1 │ species_emb │ Embedding         │    128 │ train │     0 │
│ 2 │ target_emb  │ Embedding         │     24 │ train │     0 │
│ 3 │ tabular_net │ Sequential        │  1.0 K │ train │     0 │
│ 4 │ head        │ Sequential        │  172 K │ train │     0 │
│ 5 │ loss_fn     │ HuberLoss         │      0 │ train │     0 │
└───┴─────────────┴───────────────────┴────────┴───────┴───────┘

Trainable params: 173 K                                                                                            
Non-trainable params: 633 M                                                                                        
Total params: 633 M                                                                                                
Total estimated model params size (MB): 2.5 K                                                                      
Modules in train mode: 730                                                                                         
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

`Trainer.fit` stopped: `max_epochs=1` reached.


Starting Inference...
Processed 0 images...
✅ Done


In [10]:
import torch

torch.cuda.empty_cache()

In [11]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import timm 
import os
import cv2
import numpy as np
import pytorch_lightning as pl
import gc
import pandas as pd
from torch.utils.data import Dataset, DataLoader
# from tqdm import tqdm <-- Removed to prevent IOPub timeout
from torchvision.transforms import v2
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from pytorch_lightning.callbacks import EarlyStopping, ModelCheckpoint

# --- 0. Memory Management Setup ---
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
torch.cuda.empty_cache()
gc.collect()

print("\n=== STEP 5: Final Biomass Prediction (Memory Optimized & Log Fix) ===")

# --- 1. Global Setup & Encoders ---
TARGET_NAME_LE = LabelEncoder()
# Ensure train_pd exists in your context from previous cells
TARGET_NAME_LE.fit(train_pd["target_name"].astype(str).unique())

# Helper function for safe encoding (handling unseen labels)
def safe_encode(le, val):
    val = str(val)
    if val in le.classes_:
        return le.transform([val])[0]
    return len(le.classes_) # Map unknown to a new index

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
image_root_dir = "/kaggle/input/csiro-biomass"
IMG_SIZE = 768  
BATCH_SIZE = 1  # Vital for ViT-Huge memory constraints
ACCUMULATE_GRAD = 16 # Simulates a batch size of 16

# --- 2. Transforms ---
train_tfs = v2.Compose([
    v2.ToImage(),
    v2.Resize((IMG_SIZE, IMG_SIZE)),
    v2.RandomHorizontalFlip(p=0.5),
    v2.RandomVerticalFlip(p=0.5),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

valid_tfs = v2.Compose([
    v2.ToImage(),
    v2.Resize((IMG_SIZE, IMG_SIZE)),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# --- 3. Dataset & Model ---
class BiomassDataset(Dataset):
    def __init__(self, df, root_dir, transform=None, is_train=True):
        self.df = df.reset_index(drop=True)
        self.root_dir = root_dir
        self.transform = transform
        self.is_train = is_train

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image_path = os.path.join(self.root_dir, row["image_path"])
        
        try:
            img = Image.open(image_path).convert("RGB")
        except:
            img = Image.new('RGB', (IMG_SIZE, IMG_SIZE), (0, 0, 0))
            
        if self.transform:
            img = self.transform(img)

        # Tabular features
        # Assuming STATE_LE and SPECIES_LE are defined in previous cells
        tab = torch.tensor([
            float(safe_encode(STATE_LE, row["State"])),
            float(safe_encode(SPECIES_LE, row["Species"])),
            float(row["Pre_GSHH_NDVI"]),
            float(row["Height_Ave_cm"]),
            float(safe_encode(TARGET_NAME_LE, row["target_name"]))
        ], dtype=torch.float32)

        if self.is_train:
            target = torch.tensor([row["target"]], dtype=torch.float32)
            return img, tab, target
        return img, tab

class BiomassLightningModel(pl.LightningModule):
    def __init__(self, state_dim, species_dim, target_name_dim, lr=1e-4):
        super().__init__()
        self.save_hyperparameters()

        # 1. FIXED Backbone: Forcing patch_size=16 for Dinov3
        try:
            self.backbone = timm.create_model(
                'vit_huge_patch14_224', 
                pretrained=False, 
                num_classes=0,
                img_size=IMG_SIZE,
                patch_size=16 
            )
        except:
            self.backbone = timm.create_model(
                'vit_huge_patch16_224', 
                pretrained=False, 
                num_classes=0,
                img_size=IMG_SIZE
            )

        self.backbone.set_grad_checkpointing(True)

        # 2. Weight Loading Logic
        backbone_path = "/kaggle/input/vit-huge-plus-patch16-dinov3-lvd1689m/pytorch/default/1/vit_huge_plus_patch16_dinov3.lvd1689m_backbone.pth"
        if os.path.exists(backbone_path):
            state_dict = torch.load(backbone_path, map_location='cpu')
            if 'state_dict' in state_dict: state_dict = state_dict['state_dict']
            elif 'model' in state_dict: state_dict = state_dict['model']
            
            clean_state_dict = {k.replace('backbone.', '').replace('model.', '').replace('_orig_mod.', ''): v 
                               for k, v in state_dict.items()}
            msg = self.backbone.load_state_dict(clean_state_dict, strict=False)
            print(f"✅ Final Backbone Loaded. Missing: {len(msg.missing_keys)}")

        # 3. Memory Optimization: Freeze Backbone
        for param in self.backbone.parameters():
            param.requires_grad = False

        self.img_dim = self.backbone.num_features # 1280
        self.state_emb = nn.Embedding(state_dim + 1, 8)
        self.species_emb = nn.Embedding(species_dim + 1, 16)
        self.target_name_emb = nn.Embedding(target_name_dim + 1, 8)
        
        self.tab_net = nn.Sequential(
            nn.Linear(34, 128),
            nn.LayerNorm(128), 
            nn.SiLU(), 
            nn.Dropout(0.3),
            nn.Linear(128, 64)
        )
        
        self.head = nn.Sequential(
            nn.Linear(self.img_dim + 64, 512),
            nn.LayerNorm(512),
            nn.SiLU(),
            nn.Dropout(0.4),
            nn.Linear(512, 1)
        )

    def forward(self, img, state, species, target_name, ndvi_height):
        img_feats = self.backbone(img)
        if isinstance(img_feats, (list, tuple)): img_feats = img_feats[-1]
        if img_feats.dim() > 2: img_feats = img_feats.mean(dim=1)
            
        s_emb = self.state_emb(state.long())
        sp_emb = self.species_emb(species.long())
        t_emb = self.target_name_emb(target_name.long())
        
        tab_combined = torch.cat([s_emb, sp_emb, t_emb, ndvi_height], dim=1)
        tab_feats = self.tab_net(tab_combined)
        
        combined = torch.cat([img_feats, tab_feats], dim=1)
        return self.head(combined).squeeze(1)

    def training_step(self, batch, batch_idx):
        img, tab, y = batch
        preds = self(img, tab[:,0], tab[:,1], tab[:,4], tab[:, 2:4])
        loss = F.huber_loss(preds, y.squeeze(), delta=1.0)
        # prog_bar=False to reduce logging
        self.log("train_loss", loss, prog_bar=False, on_step=False, on_epoch=True)
        return loss

    def validation_step(self, batch, batch_idx):
        img, tab, y = batch
        preds = self(img, tab[:,0], tab[:,1], tab[:,4], tab[:, 2:4])
        mse = F.mse_loss(preds, y.squeeze())
        self.log("val_mse", mse, prog_bar=False, on_step=False, on_epoch=True)

    def configure_optimizers(self):
        return torch.optim.AdamW(self.parameters(), lr=self.hparams.lr, weight_decay=0.05)

# --- 4. Execution Pipeline ---
train_df, valid_df = train_test_split(train_pd, test_size=0.15, random_state=42)

train_loader = DataLoader(BiomassDataset(train_df, image_root_dir, train_tfs), 
                          batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True, drop_last=True)
valid_loader = DataLoader(BiomassDataset(valid_df, image_root_dir, valid_tfs), 
                          batch_size=BATCH_SIZE, num_workers=2, drop_last=False)

model = BiomassLightningModel(
    state_dim=len(STATE_LE.classes_), 
    species_dim=len(SPECIES_LE.classes_), 
    target_name_dim=len(TARGET_NAME_LE.classes_)
)

# --- CRITICAL FIX: Disabled progress bar and increased log interval ---
trainer = pl.Trainer(
    accelerator="gpu", 
    devices=1, 
    max_epochs=1, 
    precision="16-mixed",
    accumulate_grad_batches=ACCUMULATE_GRAD,
    callbacks=[
        EarlyStopping(monitor="val_mse", patience=3, verbose=False), 
        ModelCheckpoint(monitor="val_mse", filename="best-biomass-huge", mode="min", verbose=False)
    ],
    log_every_n_steps=50,       # Reduced logging frequency
    enable_progress_bar=False,  # Disabled to prevent IO crash
    enable_model_summary=True
)

print("Starting Training (Progress bar disabled)...")
trainer.fit(model, train_loader, valid_loader)

# --- 5. Final Inference ---
del train_loader, valid_loader
torch.cuda.empty_cache()
gc.collect()

model.eval().to(device)
final_results = []

test_dataset = BiomassDataset(test_pd, image_root_dir, valid_tfs, is_train=False)
test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False, num_workers=2)

print("Starting Inference...")
with torch.no_grad():
    # Removed tqdm loop, using simple enumeration
    for i, batch in enumerate(test_loader):
        try:
            img, tab = batch
            img, tab = img.to(device), tab.to(device)
            pred = model(img, tab[:,0], tab[:,1], tab[:,4], tab[:, 2:4]).item()
            final_results.append(max(0.0, float(pred)))
        except Exception as e:
            final_results.append(0.0)
            
        if i % 500 == 0:
            print(f"Processed {i} samples...")

submission_df = pd.DataFrame({"sample_id": test_pd["sample_id"], "target": final_results})
submission_df.to_csv('/kaggle/working/submission.csv', index=False)
print("✅ Pipeline Complete! Submission saved.")


=== STEP 5: Final Biomass Prediction (Memory Optimized & Log Fix) ===


Using 16bit Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


✅ Final Backbone Loaded. Missing: 97
Starting Training (Progress bar disabled)...


/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/model_summary/model_summary.py:242: Precision 16-mixed is not supported by the model summary.  Estimated model size in MB will not be accurate. Using 32 bits instead.


┏━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name            ┃ Type              ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ backbone        │ VisionTransformer │  633 M │ train │     0 │
│ 1 │ state_emb       │ Embedding         │     40 │ train │     0 │
│ 2 │ species_emb     │ Embedding         │    256 │ train │     0 │
│ 3 │ target_name_emb │ Embedding         │     48 │ train │     0 │
│ 4 │ tab_net         │ Sequential        │ 13.0 K │ train │     0 │
│ 5 │ head            │ Sequential        │  690 K │ train │     0 │
└───┴─────────────────┴───────────────────┴────────┴───────┴───────┘

Trainable params: 703 K                                                                                            
Non-trainable params: 633 M                                                                                        
Total params: 634 M                                                                                                
Total estimated model params size (MB): 2.5 K                                                                      
Modules in train mode: 731                                                                                         
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

/tmp/ipykernel_24/4203664449.py:185: UserWarning: Using a target size (torch.Size([])) that is different to the input size (torch.Size([1])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  mse = F.mse_loss(preds, y.squeeze())
/tmp/ipykernel_24/4203664449.py:177: UserWarning: Using a target size (torch.Size([])) that is different to the input size (torch.Size([1])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  loss = F.huber_loss(preds, y.squeeze(), delta=1.0)
`Trainer.fit` stopped: `max_epochs=1` reached.


Starting Inference...
Processed 0 samples...
✅ Pipeline Complete! Submission saved.


In [12]:
submission_df.head()

,sample_id,target
0,ID1001187975__Dry_Clover_g,9.225654
1,ID1001187975__Dry_Dead_g,9.225846
2,ID1001187975__Dry_Green_g,9.226435
3,ID1001187975__Dry_Total_g,9.226521
4,ID1001187975__GDM_g,9.226074
